In [1]:
!pip -q install ragas langchain pandas tqdm
!pip -q install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 7.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 65.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 37.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is 

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

avalai_api_key = ""
avalai_base_url = "https://api.avalai.ir/v1"
avalai_model = "gpt-5-nano"


lc_llm = ChatOpenAI(
    openai_api_key=avalai_api_key,
    openai_api_base=avalai_base_url,
    model=avalai_model,
    temperature=0.0
)

ragas_llm = LangchainLLMWrapper(lc_llm)


/tmp/ipykernel_37/3799465664.py:9: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  lc_llm = ChatOpenAI(
/tmp/ipykernel_37/3799465664.py:16: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  ragas_llm = LangchainLLMWrapper(lc_llm)


In [3]:
import json, os
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

images_dir = "/kaggle/input/diagram2graph-dataset/diagram2graph" 
json_path = "/kaggle/input/fewshot-output/rdf_extractions_FewShot.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []
for item in data.get("dataset", []):
    img_name = item.get("source_image")
    rdf_text = item.get("rdf_graph_turtle", "")
    img_path = os.path.join(images_dir, img_name)

    if not os.path.exists(img_path):
        print("⚠️ Not Found", img_path)
        continue

    sample = SingleTurnSample(
        user_input=f"Extract RDF triples from this image ({img_name})",
        response=rdf_text,
        retrieved_contexts=[img_path],  
        metadata={"image": img_name}
    )
    samples.append(sample)

dataset = EvaluationDataset(samples=samples)
print("sample number", len(samples))


sample number 219


In [4]:
from ragas import evaluate
from ragas.metrics import MultiModalFaithfulness, MultiModalRelevance

metrics = [MultiModalFaithfulness(), MultiModalRelevance()]

results = evaluate(
    dataset,
    metrics=metrics,
    llm=ragas_llm,
    show_progress=True
)

df = results.to_pandas()
df.head()


Evaluating:   0%|          | 0/438 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,faithful_rate,relevance_rate
0,Extract RDF triples from this image (/kaggle/i...,[/kaggle/input/diagram2graph-dataset/diagram2g...,@prefix d2g: <http://example.org/diagram2graph...,1.0,1.0
1,Extract RDF triples from this image (/kaggle/i...,[/kaggle/input/diagram2graph-dataset/diagram2g...,@prefix d2g: <http://example.org/diagram2graph...,0.0,1.0
2,Extract RDF triples from this image (/kaggle/i...,[/kaggle/input/diagram2graph-dataset/diagram2g...,@prefix d2g: <http://example.org/diagram2graph...,1.0,1.0
3,Extract RDF triples from this image (/kaggle/i...,[/kaggle/input/diagram2graph-dataset/diagram2g...,@prefix d2g: <http://example.org/diagram2graph...,1.0,1.0
4,Extract RDF triples from this image (/kaggle/i...,[/kaggle/input/diagram2graph-dataset/diagram2g...,@prefix d2g: <http://example.org/diagram2graph...,1.0,1.0


In [7]:
import pandas as pd

df.to_csv("/kaggle/working/per_sample_scores.csv", index=False)

print("📂 save output per_sample_scores.csv")



📂 save output per_sample_scores.csv


In [8]:
import pandas as pd

# read csv file
df = pd.read_csv("/kaggle/working/per_sample_scores.csv")

# Calcuate Avg
summary = df[["faithful_rate", "relevance_rate"]].mean().reset_index()
summary.columns = ["metric", "mean_score"]

# save
summary.to_csv("/kaggle/working/summary.csv", index=False)

summary


,metric,mean_score
0,faithful_rate,0.648402
1,relevance_rate,0.977169
